Random Forest Independent working

Linear regression worked better, so i am rejecting this one.


In [55]:
import pandas as pd
df= pd.read_csv("train.csv")
df.shape


(1460, 81)

In [56]:
# DROPPED COLUMNS AND REASONS
dropped = {
    'PoolQC': 'only 3 houses have pool, not enough data',
    'RoofMatl': '99% same value, no variation',
    'GarageCond': 'redundant with GarageQual, lower variation',
    'Alley': 'only 91 houses have alley access, not enough data',
    'Utilities': '99% same value, no variation',
    'Heating': '99% same value, no variation',
    'LowQualFinSF': 'limited data availability',
    'PoolArea': 'limited data availability',
    'MiscFeature': 'lot of missing values ',
    'MiscVal': 'limited data availability',
    'LandSlope': '99% same value, no variation',
    'BsmtHalfBath': 'limited data availability',
    'BsmtFinSF2': 'limited data availability',
    'BsmtFinType2': '99% same value, no variation',
    'BsmtUnfSF': 'limited data availability',
    'KitchenAbvGr': '99% same value, no variation',
    'GarageCond': 'redundant with GarageQual, lower variation',
    'MoSold': ' it do not affect the price of the house',
    'YrSold': ' it do not affect the price of the house',
    'OpenPorchSF': 'combined with other porch features to create TotalPorchSF',
    'EnclosedPorch': 'combined with other porch features to create TotalPorchSF',
    '3SsnPorch': 'combined with other porch features to create TotalPorchSF',
    'ScreenPorch': 'combined with other porch features to create TotalPorchSF',
    'BsmtFinSF1':'already have total basement sf',
    'BsmtFinSF2': 'already have total basement sf',
    'BsmtUnfSF': 'already have total basement sf',
    'OverallCond': ' less impact on price compared to OverallQuall, corr in negative',
    
    
}


In [57]:
df['TotalPorchSF'] = df['OpenPorchSF'] + df['EnclosedPorch'] + df['3SsnPorch'] + df['ScreenPorch']
df.shape

(1460, 82)

In [58]:
df = df.drop(columns=list(dropped.keys()), errors='ignore')
df.shape

(1460, 58)

In [59]:
df['MSSubClass'] = df['MSSubClass'].astype(str)

In [60]:
df['Electrical'].value_counts()
df['Electrical'].dtype


df["Electrical"] = df["Electrical"].fillna(df["Electrical"].mode()[0])

In [61]:
# Filling garage-related missing values

garage_cols = ["GarageType", "GarageFinish", "GarageQual"]

for col in garage_cols:
    df[col] = df[col].fillna("NA")

df["GarageYrBlt"] = df["GarageYrBlt"].fillna(0)

In [62]:
df['BsmtExposure'].value_counts()
df['BsmtQual'].value_counts()
pd.crosstab(df["BsmtQual"], df["BsmtCond"])
df.groupby("BsmtQual")["SalePrice"].mean()
df.groupby("BsmtCond")["SalePrice"].mean()

BsmtCond
Fa    121809.533333
Gd    213599.907692
Po     64000.000000
TA    183632.620900
Name: SalePrice, dtype: float64

In [63]:
df.loc[948, "BsmtExposure"] = "No"

In [64]:
# Filling basement-related missing values

bsmt_cols = ["BsmtQual", "BsmtCond", "BsmtFinType1", "BsmtExposure"]

for col in bsmt_cols:
    df[col] = df[col].fillna("NA")

In [65]:
df['FireplaceQu']= df['FireplaceQu'].fillna("NA")

In [66]:
df['Fence']= df['Fence'].fillna("NA")

In [67]:
mode_value = df["MasVnrType"].mode()[0]

df.loc[[624, 1300, 1334], "MasVnrType"] = mode_value

In [68]:
# Filling remaining missing masonry veneer values

df["MasVnrType"] = df["MasVnrType"].fillna("None")
df["MasVnrArea"] = df["MasVnrArea"].fillna(0.0)

In [69]:
df['LotFrontage'] = df.groupby('Neighborhood')['LotFrontage'].transform(
    lambda x: x.fillna(x.median())
)

In [70]:
missing_values = df.isnull().sum()
missing_values = missing_values[missing_values > 0].sort_values(ascending=False)
print(missing_values)


Series([], dtype: int64)


In [71]:
# Assuming you have your cleaned dataframe before encoding saved as 'df_cleaned'
# If not, rerun your cleaning steps up to the point just before encoding

from sklearn.preprocessing import OrdinalEncoder
import pandas as pd

# Separate categorical and numerical columns
cat_cols = df.select_dtypes(include='object').columns.tolist()
num_cols = df.select_dtypes(exclude='object').columns.tolist()

# Apply ordinal encoding
oe = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)
df_encoded = df.copy()
df_encoded[cat_cols] = oe.fit_transform(df[cat_cols])

C:\Users\kashi\AppData\Local\Temp\ipykernel_3432\3434201261.py:8: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_cols = df.select_dtypes(include='object').columns.tolist()


In [72]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score
import numpy as np

X = df_encoded.drop(columns=['SalePrice'])
y = df_encoded['SalePrice']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# No scaling needed for RF
rf = RandomForestRegressor(n_estimators=200, max_depth=10, random_state=42)
rf.fit(X_train, y_train)

preds = rf.predict(X_test)
print("RMSE:", np.sqrt(mean_squared_error(y_test, preds)))
print("R2:", r2_score(y_test, preds))

RMSE: 28851.215682820242
R2: 0.8914788466758223


In [73]:
df['LotArea'] = np.log1p(df['LotArea'])
df['MasVnrArea']= np.log1p(df['MasVnrArea'])
df['GrLivArea']= np.log1p(df['GrLivArea'])
df['WoodDeckSF']= np.log1p(df['WoodDeckSF'])
df['LotFrontage']= np.log1p(df['LotFrontage'])



In [74]:
df = df.drop(columns=['1stFlrSF', '2ndFlrSF'])
df.shape

(1460, 56)

In [75]:
df['TotalBath'] = df['FullBath'] + df['HalfBath'] + df['BsmtFullBath']


In [76]:
df=df.drop(columns=['FullBath', 'HalfBath', 'BsmtFullBath'])


In [77]:
df = df.drop(columns=['BedroomAbvGr'])

In [78]:
df = df[df['SalePrice'] <= 350000]
print(df.shape)

(1406, 53)


In [79]:
# Assuming you have your cleaned dataframe before encoding saved as 'df_cleaned'
# If not, rerun your cleaning steps up to the point just before encoding

from sklearn.preprocessing import OrdinalEncoder
import pandas as pd

# Separate categorical and numerical columns
cat_cols = df.select_dtypes(include='object').columns.tolist()
num_cols = df.select_dtypes(exclude='object').columns.tolist()

# Apply ordinal encoding
oe = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)
df_encoded = df.copy()
df_encoded[cat_cols] = oe.fit_transform(df[cat_cols])

C:\Users\kashi\AppData\Local\Temp\ipykernel_3432\3434201261.py:8: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_cols = df.select_dtypes(include='object').columns.tolist()


In [80]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score
import numpy as np

X = df_encoded.drop(columns=['SalePrice'])
y = df_encoded['SalePrice']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# No scaling needed for RF
rf = RandomForestRegressor(n_estimators=200, max_depth=10, random_state=42)
rf.fit(X_train, y_train)

preds = rf.predict(X_test)
print("RMSE:", np.sqrt(mean_squared_error(y_test, preds)))
print("R2:", r2_score(y_test, preds))

RMSE: 22364.937972697804
R2: 0.8419536735146047
